# Comparing W&B Artifacts vs MLflow Model Registry for model lineage

Model lineage — the ability to trace a deployed model back to its training data, hyperparameters, and code — is a cornerstone of reproducible ML. Two widely adopted tools address this from different angles: **W&B Artifacts** tracks data and model files as versioned, lineage-linked objects within a run, while **MLflow Model Registry** provides a centralized repository for registering, versioning, and transitioning model stages.

This notebook compares the two approaches across the dimensions that matter most for lineage: how each records provenance, how versions are managed, and how models move from experiment to deployment.

## Purpose

Understand the trade-offs between W&B Artifacts and MLflow Model Registry when building a model lineage tracking strategy. This notebook is a reference guide for practitioners evaluating which tool fits their lineage requirements.

## When to use each

| Criterion | W&B Artifacts | MLflow Model Registry |
|---|---|---|
| Primary role | Run-level artifact tracking with automatic lineage | Centralized model versioning and stage transitions |
| Lineage scope | Links artifacts to the run that produced them | Links registered models to the run that logged them |
| Versioning | Implicit via artifact versions tied to runs | Explicit version numbers with stage transitions (Staging to Production to Archived) |
| Deployment | Artifacts downloaded for local or cloud serving | Models promoted through stages; deployment hooks available |
| Team workflow | Shared runs and projects in a W&B dashboard | Shared registry with role-based access and annotations |

Neither tool is a replacement for the other; they solve overlapping but distinct problems. Teams that use both often log artifacts to W&B for experiment tracking and register models in MLflow for deployment governance.

## Steps — W&B Artifacts approach

W&B Artifacts attach files (model weights, datasets) to a run and automatically record the run context (code hash, git commit, parameters). The artifact lifecycle is:

1. **Log** — call `wandb.Artifact(name, type="model")` and `artifact.add_file(path)` to attach a model file.
2. **Link** — `run.log_artifact(artifact)` ties the artifact to the current run, recording the run ID, config, and summary metrics as provenance.
3. **Version** — uploading the same artifact name creates a new version; W&B deduplicates identical content hashes.
4. **Use** — downstream runs can reference the artifact via `wandb.use_artifact(artifact)` to pull the file and record the dependency in the new run lineage graph.

The lineage graph is visible in the W&B UI: each artifact shows which runs produced it and which runs consumed it.

In [ ]:
import wandb

# Log a model as an artifact — this records the run context automatically
run = wandb.init(project="lineage-comparison", job_type="train")

artifact = wandb.Artifact("mlp-model", type="model")
artifact.add_file("outputs/model.pt")
run.log_artifact(artifact)
run.finish()

## Steps — MLflow Model Registry approach

MLflow Model Registry provides a dedicated server-side store for model versions. The lineage flow is:

1. **Log model** — `mlflow.sklearn.log_model(model, "model")` writes the model to the tracking URI and records the run ID, params, and metrics.
2. **Register** — `mlflow.register_model("runs:/<run_id>/model", "my-model")` creates a registered model entry linked to the source run.
3. **Version** — each registration creates a new version number; the model version stores the source run ID, artifact URI, and creation metadata.
4. **Transition** — a version can be moved between stages (None to Staging to Production to Archived) with annotations describing the reason for the transition.

The registry API allows querying all versions of a model and filtering by stage, making it straightforward to audit which model version is currently deployed.

In [ ]:
import mlflow

# Log and register a model — the run ID links the model to its training context
with mlflow.start_run(run_name="mlp-training") as run:
    mlflow.sklearn.log_model(model, "model")
    result = mlflow.register_model(
        f"runs:/{run.info.run_id}/model",
        "my-model"
    )
    print(f"Registered version: {result.version}")

## Verify — checking lineage in each system

The key verification step is confirming that the lineage chain is intact: a deployed model can be traced back to the exact training run, dataset, and code version that produced it.

For W&B Artifacts, this means checking the artifact lineage graph in the UI or via the API to confirm the producing and consuming runs are recorded.

For MLflow Model Registry, this means querying the registered model source run ID and confirming the associated metrics and parameters are still accessible.

In [ ]:
# Verify W&B artifact lineage
artifact = wandb.Artifact("mlp-model", type="model")
api = wandb.Api()
run = api.run("entity/project/run-id")
for art in run.logged_artifacts():
    print(f"Artifact: {art.name} v{art.version} — used by: {art.used_by}")

# Verify MLflow model registry lineage
client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions('name = "my-model"')
for v in versions:
    print(f"Version {v.version}: source run={v.source}, stage={v.current_stage}")

## Summary

| Aspect | W&B Artifacts | MLflow Model Registry |
|---|---|---|
| Lineage granularity | Run-level, automatic | Run-level via source tracking, explicit registration |
| Versioning model | Implicit (artifact versions) | Explicit version numbers |
| Stage management | Not built-in | Staging to Production to Archived |
| UI for lineage | Built-in artifact graph | Registry UI with version history |

Choose W&B Artifacts when the priority is capturing lineage automatically as part of the experiment loop. Choose MLflow Model Registry when the priority is governing model promotions through explicit stages. Many setups use both: W&B for experiment tracking and artifact logging, MLflow for registry and deployment governance.